다양한 임베딩 모델을 평가하는 구체적인 방법을 알려드리도록 하겠습니다
- 임베딩 후보 리스트 준비 (OpenAI, Cohere, e5-base-v2)
- 활용하고자 하는 데이터셋을 임베딩 변환
- Test set 랜덤 선별 후 평가 지표 생성

---

In [3]:
!pip install cohere gensim numpy pandas scikit-learn matplotlib seaborn

zsh:1: /Users/igyeongseob/Develop/ai/RAG/vectorDB/vector/venv/bin/pip: bad interpreter: /Users/igyeongseob/Develop/ai/vector/venv/bin/python3: no such file or directory
  Using cached cohere-5.20.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached gensim-4.4.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (8.4 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached fastavro-1.12.1-cp312-cp312-macosx_10_13_universal2.whl.metadata (5.8 kB)
  Using cached smart_open-7.5.0-py3-none-any.whl.metadata (24 kB)
Using cached fastavro-1.12.1-cp312-cp312-macosx_10_13_universal2.whl (1.0 MB)
Using cached gensim-4.4.0-cp312-cp312-macosx_11_0_arm64.whl (24.5 MB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Using cached smart_open-7.5.0-py3-none-any.whl (63 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [cohere]2m4/5 [cohere]]


In [1]:
import pandas as pd
import os
import random
import cohere
import torch
import numpy as np
from transformers import AutoModel, AutoTokenizer
import openai
from openai import OpenAI
from tqdm.notebook import tqdm
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# initialize openai
from dotenv import load_dotenv
load_dotenv()
openai.api_key = os.environ["OPENAI_API_KEY"]

# initialize cohere
co = cohere.Client()

import warnings
warnings.filterwarnings('ignore')


### Read dataset

In [ ]:
df = pd.read_csv("dataset/abcnews.csv")

In [ ]:
df

,publish_date,headline_text
0,20030219,aba decides against community broadcasting lic...
1,20030219,act fire witnesses must be aware of defamation
2,20030219,a g calls for infrastructure protection summit
3,20030219,air nz staff in aust strike for pay rise
4,20030219,air nz strike to affect australian travellers
...,...,...
1244179,20211231,two aged care residents die as state records 2...
1244180,20211231,victoria records 5;919 new cases and seven deaths
1244181,20211231,wa delays adopting new close contact definition
1244182,20211231,western ringtail possums found badly dehydrate...


In [ ]:
df.head()

,publish_date,headline_text
0,20030219,aba decides against community broadcasting lic...
1,20030219,act fire witnesses must be aware of defamation
2,20030219,a g calls for infrastructure protection summit
3,20030219,air nz staff in aust strike for pay rise
4,20030219,air nz strike to affect australian travellers


### 1. Playground

In [ ]:
text1 = df.loc[2, 'headline_text']
print(text1)

a g calls for infrastructure protection summit


In [ ]:
text2 = df.loc[3, 'headline_text']
print(text2)

air nz staff in aust strike for pay rise


In [ ]:
def create_embeddings(txt_list, provider='openai'):
    if provider=='openai':
        client = OpenAI()

        response = client.embeddings.create(
        input=txt_list,
        model="text-embedding-3-small")
        responses = [r.embedding for r in response.data]

        return responses
    
    elif provider=='cohere':
        doc_embeds = co.embed(
        txt_list,
        input_type="search_document",
        model="embed-english-v3.0")
        return doc_embeds.embeddings
    else:
        assert False, "Double check provider name"

In [ ]:
emb1 = create_embeddings(df.loc[2, 'headline_text'])
emb2 = create_embeddings(df.loc[3, 'headline_text'])

In [ ]:
from utils import cosine_similarity

In [ ]:
# simarity between two embeddings
print("Cosine 유사도 : {}.\n사용된 문장 : \n{}\n{}".format(cosine_similarity(emb1[0], emb2[0]), text1, text2))

Cosine 유사도 : 0.16141813010923994.
사용된 문장 : 
a g calls for infrastructure protection summit
air nz staff in aust strike for pay rise


In [ ]:
text3 = df.loc[4, 'headline_text']

emb3 = create_embeddings(text3)
print("Cosine 유사도 : {}.\n사용된 문장 : \n{}\n{}".format(cosine_similarity(emb1[0], emb3[0]), text1, text3))

Cosine 유사도 : 0.15100410380900078.
사용된 문장 : 
a g calls for infrastructure protection summit
air nz strike to affect australian travellers


In [ ]:
text4 = df.loc[6, 'headline_text']

emb3 = create_embeddings(text4)
print("Cosine 유사도 : {}.\n사용된 문장 : \n{}\n{}".format(cosine_similarity(emb1[0], emb3[0]), text1, text4))

Cosine 유사도 : 0.1365442494556233.
사용된 문장 : 
a g calls for infrastructure protection summit
antic delighted with record breaking barca


---

### 2. Embedding vector Dataset 만들기

openai embeddings

In [ ]:
# create embeddings (openai)
# (비용 발생 주의)
MAX_ITEMS = 10 # 원하는 개수 제한

texts = (
    df["headline_text"]
    .dropna()
    .astype(str)
    .head(MAX_ITEMS)
    .tolist()
)

openai_emb = create_embeddings(
    texts,
    provider="openai"
)

In [ ]:
texts

['aba decides against community broadcasting licence',
 'act fire witnesses must be aware of defamation',
 'a g calls for infrastructure protection summit',
 'air nz staff in aust strike for pay rise',
 'air nz strike to affect australian travellers',
 'ambitious olsson wins triple jump',
 'antic delighted with record breaking barca',
 'aussie qualifier stosur wastes four memphis match',
 'aust addresses un security council over iraq',
 'australia is locked into war timetable opp']

In [ ]:
# df['openai_emb'] = openai_emb
openai_emb

[[0.02436016872525215,
  0.010141161270439625,
  0.07135467231273651,
  0.0008161420118995011,
  -0.02599436044692993,
  0.025856906548142433,
  -0.0004944579559378326,
  0.013134635984897614,
  0.011928082443773746,
  0.03512751683592796,
  0.02159578539431095,
  -0.05113344267010689,
  -0.032225675880908966,
  0.05247745290398598,
  0.04377193748950958,
  0.005280581768602133,
  0.024253258481621742,
  -0.0006300043314695358,
  -0.04502430930733681,
  0.008254965767264366,
  -0.015234651044011116,
  0.021580513566732407,
  -0.013745549134910107,
  -0.013455365784466267,
  -0.030148571357131004,
  -0.03732680156826973,
  -0.027032913640141487,
  -0.00833896640688181,
  0.028957290574908257,
  -0.0034707512240856886,
  0.03772389516234398,
  -0.053852006793022156,
  -0.01617392897605896,
  0.021656876429915428,
  -0.018296852707862854,
  -0.0016093747690320015,
  -0.0024474714882671833,
  -0.026269271969795227,
  0.001305827172473073,
  -0.044107940047979355,
  -0.027689645066857338,
 

cohere embeddings

In [ ]:
# create embeddings (cohere)
# (비용 발생 주의)
cohere_emb = create_embeddings(df.text.tolist(), 'cohere')

AttributeError: 'DataFrame' object has no attribute 'text'

In [ ]:
# df['cohere_emb'] = cohere_emb

e5 embeddings

In [ ]:
# load gpu if possible
device = "cuda" if torch.cuda.is_available() else "cpu"

model_id = "intfloat/e5-base-v2"

# init tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id).to(device)
model.eval()

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [ ]:
# def create_e5_emb(docs, model):
#     """
#     e5 embedding 모델을 활용하여 임베딩 벡터 생성
#     """
#     docs = [f"query: {d}" for d in docs]
#     # tokenize
#     tokens = tokenizer(
#         docs, padding=True, max_length=512, truncation=True, return_tensors="pt"
#     ).to(device)
#     with torch.no_grad():
#         out = model(**tokens)
#         last_hidden = out.last_hidden_state.masked_fill( # from last hidden state
#             ~tokens["attention_mask"][..., None].bool(), 0.0
#         )
#         # average out embeddings per token (non-padding)
#         doc_embeds = last_hidden.sum(dim=1) / tokens["attention_mask"].sum(dim=1)[..., None]
#     return doc_embeds.cpu().numpy()

긴 runtime 주의 (약 2시간)

In [ ]:
# data = df.text.tolist()
# batch_size = 128

# for i in tqdm(range(0, len(data), batch_size)):
#     i_end = min(len(data), i+batch_size)
#     data_batch = data[i:i_end]
#     # embed current batch
#     embed_batch = create_e5_emb(data_batch)
#     if i == 0:
#         emb3 = embed_batch.copy()
#     else:
#         emb3 = np.concatenate([emb3, embed_batch.copy()])

In [ ]:
# emb3 = [list(e) for e in emb3]
# df['e5_emb'] = emb3

In [ ]:
# df.to_csv("quora_dataset_emb.csv", index=False)

embedding이 이미 처리된 데이터 읽어오기

In [4]:
df = pd.read_csv("dataset/quora_dataset_emb.csv")
# str -> list 형태로 변환
import json
df['openai_emb'] = df['openai_emb'].apply(json.loads)
df['cohere_emb'] = df['cohere_emb'].apply(json.loads)
df['e5_emb'] = df['e5_emb'].apply(json.loads)
df['duplicated_questions'] = df['duplicated_questions'].apply(json.loads)

In [5]:
df.head()

,text,id,duplicated_questions,length,openai_emb,cohere_emb,e5_emb
0,Astrology: I am a Capricorn Sun Cap moon and c...,11,[12],1,"[-0.005765771958976984, -0.018585262820124626,...","[-0.05834961, -0.010795593, -0.04522705, 0.035...","[0.059878636, -0.15769655, -0.14131568, -0.546..."
1,"I'm a triple Capricorn (Sun, Moon and ascendan...",12,[11],1,"[0.026014558970928192, -0.014319832436740398, ...","[-0.022338867, -0.0063285828, -0.057128906, 0....","[0.08937627, -0.2954505, -0.33455396, -0.32940..."
2,How can I be a good geologist?,15,[16],1,"[0.005276682320982218, 0.004194203298538923, 0...","[-0.012535095, 0.005092621, -0.033233643, -0.0...","[0.0825816, -0.09264662, -0.78053623, -0.32416..."
3,What should I do to be a great geologist?,16,[15],1,"[0.015116829425096512, 0.0010464431252330542, ...","[-0.013465881, 0.0018148422, -0.052612305, 0.0...","[-0.1653303, 0.19044468, -0.8906647, -0.364357..."
4,How do I read and find my YouTube comments?,23,[24],1,"[0.03505030274391174, -0.0010134828044101596, ...","[-0.0047836304, 0.028137207, -0.037231445, -0....","[0.50644577, -0.62657785, -0.2523397, -0.17112..."


### 3. Test set 선별

테스팅을 위해 필요한 랜덤 질문들 선별

In [6]:
# now choose random 10 rows of answers
test_query = random.choices(df.id, k=1000)

In [7]:
test_query[:5]

[np.int64(4182),
 np.int64(5217),
 np.int64(5121),
 np.int64(5851),
 np.int64(5702)]

In [8]:
test = df.loc[df.id.isin(test_query)]

각 테스트 질문별로 가장 유사한 질문들 top-k개 retrieve

In [9]:
from sklearn.metrics.pairwise import cosine_similarity

def search_top_k(search_df, search_df_column, id, topk):
    """
    search_df : search를 할 대상 dataframe
    search_df_column : search를 위해 사용될 embedding column name
    id : test query id
    topk : 유사도 기반으로 top-k개 선별
    """
    query = search_df.loc[search_df['id']==id, search_df_column].values[0]
    query_reshaped = np.array(query).reshape(1, -1)
    
    search_df = search_df.loc[search_df['id']!=id]
    # cosine similarity in batch
    similarities = cosine_similarity(query_reshaped, np.vstack(search_df[search_df_column].values)).flatten()
    
    search_df['similarity'] = similarities
    
    # Get top-k indices
    # hence we sort the topk indices again to ensure they are truly the top-k
    topk_indices = np.argpartition(similarities, -topk)[-topk:]
    topk_indices_sorted = topk_indices[np.argsort(-similarities[topk_indices])]
    
    # Retrieve the top-k results
    search_result = search_df.iloc[topk_indices_sorted]
    
    return search_result


- 각 테스트 질문당 데이터 전체를 대상으로 cosine_similarity를 계산하고
- openai embedding, cohere embedding에 대해 각각 질문 k 개씩 진행
- search_result format :
```json
{
    'question id' : cosine_sim 기준 유사한 질문 top-k개를 담은 pd.DataFrame,
    'question id' : ...
}
```

In [10]:
# 각 질문들 중, test 질문과 동일한 질문이 가장 유사하게 도출될 것이기 때문에
# test 질문을 제외한 top-5
query_results_openai = { k:search_top_k(df, 'openai_emb', k, 5) for k in test.id }
query_results_cohere = { k:search_top_k(df, 'cohere_emb', k, 5) for k in test.id }
# query_results_e5 = { k:search_top_k(df, 'e5_emb', k, 5) for k in test.id }

테스트 결과 엿보기

In [12]:
test.loc[test.length==3].tail()

,text,id,duplicated_questions,length,openai_emb,cohere_emb,e5_emb
5102,How do I prepare for the IAS exam at home?,13689,"[13690, 14412, 77]",3,"[-0.026480073109269142, -0.0040580169297754765...","[-0.036315918, 0.037841797, -0.008079529, 0.00...","[-0.22943993, -0.31133857, -0.7906301, -0.5486..."
5265,How do I tell if a girl I sit next to likes me?,14160,"[10062, 14159, 10061]",3,"[-0.033233653753995895, -0.05941774323582649, ...","[-0.02444458, 0.020690918, -0.048980713, 0.043...","[-0.14724778, -0.9844855, -0.5622967, -0.37356..."
5345,What workout attire would guys wear in the sum...,14383,"[14382, 7092, 7091]",3,"[0.015372652560472488, 0.05050443112850189, -0...","[-0.041809082, -0.008651733, -0.10015869, 0.04...","[0.100323245, -0.7722157, -0.9635854, 0.348478..."
5404,How do I my increase memory power?,14637,"[14636, 10681, 10682]",3,"[0.022645525634288788, -0.005303962621837854, ...","[0.0047340393, 0.02658081, -0.052825928, -0.02...","[-0.31909862, -0.46033445, -0.46627238, -0.031..."
5419,Do you think there's life on other planets?,14686,"[12523, 12524, 14685]",3,"[0.01339676696807146, 0.008889188058674335, -0...","[-0.032592773, 0.010681152, -0.022476196, 0.04...","[-0.07575863, -0.05358007, -0.8569687, -0.0457..."


In [13]:
test.loc[test['id']==14182, 'text'].values

array([], dtype=object)

In [14]:
query_results_openai[14160]

,text,id,duplicated_questions,length,openai_emb,cohere_emb,e5_emb,similarity
3859,How do I know if this girl likes me?,10062,"[14160, 14159, 10061]",3,"[0.005086907651275396, -0.04189901426434517, 0...","[-0.054016113, 0.020217896, -0.054351807, 0.03...","[0.15249164, -0.9194247, -0.5637354, -0.653950...",0.716677
5264,I really like this girl. How can I tell if she...,14159,"[10062, 14160]",2,"[0.02182772010564804, -0.0494832806289196, 0.0...","[-0.035308838, 0.011474609, -0.06335449, 0.023...","[0.11682795, -1.0554899, -0.6295922, -0.749011...",0.671997
3858,How can a boy know if a girl likes him or not?,10061,"[10062, 14160]",2,"[0.014881503768265247, -0.0225174929946661, -0...","[-0.033721924, 0.013214111, -0.05218506, 0.026...","[0.05057525, -0.5991401, -0.77512205, -0.24307...",0.572684
1915,How do I tell if a guy likes me?,4994,"[2364, 4993, 2365]",3,"[-0.011051975190639496, -0.02304232493042946, ...","[-0.0023727417, 0.040924072, -0.052215576, 0.0...","[0.13690588, -0.6286884, -0.49419472, -0.29673...",0.560749
897,How do I know that a guy likes you?,2364,"[4993, 2365, 4994]",3,"[-0.007538170553743839, -0.012518931180238724,...","[-0.0055503845, 0.033416748, -0.046875, 0.0051...","[0.09061116, -0.58743984, -0.35238704, -0.2609...",0.524444


### 4. Scoring function 정의

- 각 질문별로 accuracy score 부여
    - Accuracy score : 현재 유사하다고 태그된 질문들 중 몇 개가 실제 유사한 질문들인가?

In [15]:
def score_accuracy(full_df, tmp_df, test_id):
    """
    각 테스트 질문과 유사하다고 판단된 질문들 중, 실제 duplicated_questions에 들어있는 질문들을 count
    """
    duplicated_questions = full_df.loc[full_df['id'] == test_id, 'duplicated_questions'].values[0]

    # 본인 ID는 제외
    filtered_df = tmp_df[tmp_df['id'] != test_id]
    # 현재 retrieve 해온 ID들이, 테스트 질문 내에 들어있는 아이디들인지 count
    match_count = filtered_df['id'].isin(duplicated_questions).sum()

    # Calculate the accuracy in terms of percentage
    if filtered_df.shape[0]<len(duplicated_questions):
        percentage = (match_count / filtered_df.shape[0])
    else:
        percentage = (match_count / len(duplicated_questions))
    return percentage


In [16]:
accuracy_openai = [score_accuracy(df, query_results_openai[i], i) for i in query_results_openai.keys()]
# accuracy_cohere = [score_accuracy(df, query_results_cohere[i], i) for i in query_results_cohere.keys()]
# accuracy_e5 = [score_accuracy(df, query_results_e5[i], i) for i in query_results_e5.keys()]

In [17]:
accuracy_cohere = [score_accuracy(df, query_results_cohere[i], i) for i in query_results_cohere.keys()]

In [18]:
accuracy_openai

[np.float64(1.0),
 np.float64(0.5),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(0.8),
 np.float64(0.6),
 np.float64(0.6),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(0.6),
 np.float64(0.4),
 np.float64(0.6666666666666666),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(0.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(0.8),
 np.float64(0.75),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(0.0),
 np.float64(0.6),
 np.float64(1.0),
 np.float64(

In [19]:
np.mean(accuracy_openai)

np.float64(0.9536105032822756)

In [ ]:
# np.mean(accuracy_cohere)

In [ ]:
# np.mean(accuracy_e5)

오답 엿보기

In [20]:
indices = [[index,value] for index, value in enumerate(accuracy_openai) if value <= 0.5]

In [21]:
indices

[[1, np.float64(0.5)],
 [11, np.float64(0.4)],
 [19, np.float64(0.0)],
 [51, np.float64(0.0)],
 [103, np.float64(0.0)],
 [120, np.float64(0.4)],
 [138, np.float64(0.4)],
 [139, np.float64(0.4)],
 [185, np.float64(0.5)],
 [205, np.float64(0.2)],
 [292, np.float64(0.4)],
 [294, np.float64(0.3333333333333333)],
 [348, np.float64(0.0)],
 [353, np.float64(0.4)],
 [443, np.float64(0.0)],
 [444, np.float64(0.2)],
 [462, np.float64(0.4)],
 [521, np.float64(0.0)],
 [565, np.float64(0.25)],
 [572, np.float64(0.0)],
 [616, np.float64(0.0)],
 [658, np.float64(0.2)],
 [675, np.float64(0.2)],
 [731, np.float64(0.5)],
 [764, np.float64(0.4)],
 [797, np.float64(0.4)],
 [799, np.float64(0.0)],
 [875, np.float64(0.0)],
 [906, np.float64(0.4)],
 [907, np.float64(0.2)],
 [913, np.float64(0.4)]]

In [ ]:
list(query_results_openai.keys())[60]

817

In [85]:
test.loc[test['id']==985]

,text,id,duplicated_questions,length,openai_emb,cohere_emb,e5_emb
378,What are some tricks to study effectively?,985,[984],1,"[0.022660082206130028, -0.025483181700110435, ...","[0.012863159, 0.02041626, -0.017868042, 0.0209...","[0.2421261, -0.087377824, -0.7994292, -0.37665..."


In [86]:
test.loc[test['id']==984]

,text,id,duplicated_questions,length,openai_emb,cohere_emb,e5_emb


In [87]:
query_results_openai[985]

,text,id,duplicated_questions,length,openai_emb,cohere_emb,e5_emb,similarity
2674,What are some good tips for self study?,7033,[7034],1,"[0.011717529967427254, -0.036956727504730225, ...","[-0.004623413, 0.0098724365, -0.028701782, 0.0...","[0.042563524, -0.11200364, -0.805879, -0.17195...",0.709618
2675,What are some tips for self study?,7034,[7033],1,"[0.013052663765847683, -0.036678362637758255, ...","[-0.0050735474, 0.015113831, -0.035949707, 0.0...","[0.0115949465, -0.0983381, -0.8623934, -0.2426...",0.696768
4327,What are the best ways to study/memorize things?,11331,[11332],1,"[-0.01611529476940632, -0.020404629409313202, ...","[-0.018630981, -0.0075263977, -0.024902344, -0...","[0.3084511, 0.2693039, -0.69890785, -0.1166682...",0.661062
1456,How do I concentrate better in my studies?,3902,[3903],1,"[0.008100658655166626, -0.02210518717765808, -...","[0.010749817, 0.043182373, -0.024032593, -0.00...","[0.48886126, -0.24943542, -0.675636, -0.476426...",0.646608
2961,How do I study well without getting distracted?,7803,[7802],1,"[0.0016504509840160608, -0.03981846570968628, ...","[0.041503906, 0.03475952, -0.02571106, -0.0210...","[0.45737946, -0.23788102, -0.9406135, -0.41269...",0.645310


#### 결론

- cohere, openai, e5 모두 굉장히 성능이 좋기 때문에 대부분의 task에 곧바로 활용해도 무방함.
- Local embedding 모델을 활용하고자 할 때 위와 같은 방법으로 classification 성능 & 자원 할당 체크 필요.
- 성능 평가 방법
    - 태깅된 데이터 셋 활용
    - 정성적 평가
        - 데이터 태깅을 할 노동력이 부족할 때
        - 태깅을 하기 애매한 분야 (정답이 없는 경우)

--END--